# EDA: PlantVillage

Датасет болезней растений — 38 классов (14 культур, здоровые и поражённые листья).  
Анализируем структуру, баланс классов, свойства изображений и цветовые характеристики.

**Предварительно**: скачать датасет:
```bash
python -m scripts.download_datasets --only plantvillage
```

## 0. Настройка

In [ ]:
import warnings
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datasets import load_from_disk

warnings.filterwarnings("ignore")

DATA_DIR    = Path("data/plantvillage")
SAMPLE_N    = 50000   # сколько изображений сэмплировать для тяжёлых вычислений (размеры, цвет)
SEED        = 42
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Загрузка данных

In [ ]:
raw = load_from_disk(str(DATA_DIR))

# Датасет может быть DatasetDict (с именованными сплитами) или просто Dataset
if hasattr(raw, "keys"):
    splits = list(raw.keys())
    print(f"DatasetDict, сплиты: {splits}")
    for s in splits:
        print(f"  {s}: {len(raw[s]):,} образцов")
    ds = raw[splits[0]]  # берём первый (обычно единственный) сплит
else:
    print("Единый Dataset (без named splits)")
    ds = raw

N = len(ds)
print(f"\nВсего образцов : {N:,}")
print(f"Признаки       : {ds.features}")

In [ ]:
CLASS_NAMES = ds.features["label"].names
N_CLASSES   = len(CLASS_NAMES)

print(f"Классов: {N_CLASSES}")
print()
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i:2d}  {name}")

## 2. Распределение классов

In [ ]:
labels = ds["label"]
counts = Counter(labels)

df = pd.DataFrame({
    "class_id":   list(counts.keys()),
    "class_name": [CLASS_NAMES[i] for i in counts.keys()],
    "count":      list(counts.values()),
}).sort_values("count", ascending=False).reset_index(drop=True)

print(f"Макс. класс : {df.iloc[0]['class_name']}  ({df.iloc[0]['count']:,})")
print(f"Мин. класс  : {df.iloc[-1]['class_name']}  ({df.iloc[-1]['count']:,})")
print(f"Среднее     : {df['count'].mean():.0f}")
print(f"Медиана     : {df['count'].median():.0f}")
print(f"Std         : {df['count'].std():.0f}")
print()
print(df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ["#e74c3c" if "healthy" not in name else "#2ecc71" for name in df["class_name"]]
bars = ax.bar(range(len(df)), df["count"], color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(df["count"].mean(), color="navy", linestyle="--", linewidth=1.2, label=f"Среднее ({df['count'].mean():.0f})")

ax.set_xticks(range(len(df)))
ax.set_xticklabels(df["class_name"], rotation=90, fontsize=7)
ax.set_ylabel("Количество изображений")
ax.set_title("Распределение классов PlantVillage (сортировка по убыванию)")

healthy_patch  = mpatches.Patch(color="#2ecc71", label="Здоровые")
diseased_patch = mpatches.Patch(color="#e74c3c", label="Больные")
ax.legend(handles=[healthy_patch, diseased_patch, ax.get_lines()[0]], loc="upper right")

plt.tight_layout()
plt.show()

## 3. Растения и болезни

Имена классов имеют вид `Plant___Disease` (двойное подчёркивание как разделитель).  
Разбираем их на компоненты.

In [ ]:
def parse_class(name: str):
    parts = name.split("___", maxsplit=1)
    plant   = parts[0].replace("_", " ")
    disease = parts[1].replace("_", " ") if len(parts) > 1 else "Unknown"
    is_healthy = "healthy" in disease.lower()
    return plant, disease, is_healthy

df[["plant", "disease", "is_healthy"]] = df["class_name"].apply(
    lambda x: pd.Series(parse_class(x))
)

print(f"Уникальных растений: {df['plant'].nunique()}")
print(f"Уникальных болезней: {df['disease'].nunique()}")
print(f"Здоровых классов   : {df['is_healthy'].sum()}")
print(f"Больных классов    : {(~df['is_healthy']).sum()}")
print()
print(df[["plant", "disease", "count", "is_healthy"]].to_string(index=False))

In [ ]:
plant_stats = df.groupby("plant").agg(
    n_classes=("class_name", "count"),
    n_diseased=("is_healthy", lambda x: (~x).sum()),
    total_images=("count", "sum"),
    healthy_images=("count", lambda x: x[df.loc[x.index, "is_healthy"]].sum()),
).sort_values("total_images", ascending=False)

plant_stats["diseased_images"] = plant_stats["total_images"] - plant_stats["healthy_images"]
plant_stats["pct_healthy"] = (plant_stats["healthy_images"] / plant_stats["total_images"] * 100).round(1)

print(plant_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Левый: изображений на растение
ax = axes[0]
plants_sorted = plant_stats.sort_values("total_images", ascending=True)
ax.barh(plants_sorted.index, plants_sorted["diseased_images"], color="#e74c3c", label="Больные")
ax.barh(plants_sorted.index, plants_sorted["healthy_images"],
        left=plants_sorted["diseased_images"], color="#2ecc71", label="Здоровые")
ax.set_xlabel("Количество изображений")
ax.set_title("Изображений на растение")
ax.legend()

# Правый: классов (болезней) на растение
ax = axes[1]
plants_sorted2 = plant_stats.sort_values("n_classes", ascending=True)
ax.barh(plants_sorted2.index, plants_sorted2["n_diseased"], color="#e74c3c", label="Болезни")
ax.barh(plants_sorted2.index, [1] * len(plants_sorted2),
        left=plants_sorted2["n_diseased"], color="#2ecc71", label="Здоровый класс")
ax.set_xlabel("Количество классов")
ax.set_title("Классов на растение")
ax.legend()

plt.suptitle("PlantVillage: распределение по растениям", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Баланс: здоровые vs больные

In [ ]:
healthy_total  = df.loc[df["is_healthy"],  "count"].sum()
diseased_total = df.loc[~df["is_healthy"], "count"].sum()

print(f"Здоровые  : {healthy_total:,}  ({healthy_total/N*100:.1f}%)")
print(f"Больные   : {diseased_total:,}  ({diseased_total/N*100:.1f}%)")
print(f"Всего     : {N:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(
    [healthy_total, diseased_total],
    labels=[f"Здоровые\n{healthy_total:,}", f"Больные\n{diseased_total:,}"],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
axes[0].set_title("Общее соотношение")

# Per-plant balance
ax = axes[1]
plants_pct = plant_stats.sort_values("pct_healthy", ascending=True)
bars = ax.barh(plants_pct.index, plants_pct["pct_healthy"], color="#2ecc71", alpha=0.8)
ax.axvline(healthy_total / N * 100, color="navy", linestyle="--", linewidth=1,
           label=f"Среднее ({healthy_total/N*100:.1f}%)")
ax.set_xlabel("% здоровых изображений")
ax.set_title("Доля здоровых по растениям")
ax.set_xlim(0, 100)
ax.legend()

plt.suptitle("Баланс здоровых / больных", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Примеры изображений

In [ ]:
# Одно случайное изображение на класс (в порядке class_id)
label_to_idx = {}
for idx, label in enumerate(labels):
    if label not in label_to_idx:
        label_to_idx[label] = idx
    if len(label_to_idx) == N_CLASSES:
        break

sorted_class_ids = sorted(label_to_idx.keys())

ncols = 8
nrows = (N_CLASSES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2.2))
axes = axes.flatten()

for i, class_id in enumerate(sorted_class_ids):
    img = ds[label_to_idx[class_id]]["image"]
    plant, disease, is_healthy = parse_class(CLASS_NAMES[class_id])
    color = "#2ecc71" if is_healthy else "#e74c3c"
    axes[i].imshow(img)
    axes[i].set_title(f"{plant}\n{disease}", fontsize=6, color=color)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

healthy_p  = mpatches.Patch(color="#2ecc71", label="Здоровые")
diseased_p = mpatches.Patch(color="#e74c3c", label="Больные")
fig.legend(handles=[healthy_p, diseased_p], loc="lower right", fontsize=9)
fig.suptitle("По одному изображению из каждого класса PlantVillage", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Томат: здоровый vs все болезни
tomato_classes = df[df["plant"].str.lower().str.contains("tomato")].sort_values("is_healthy", ascending=False)

fig, axes = plt.subplots(2, len(tomato_classes) // 2 + len(tomato_classes) % 2,
                         figsize=(16, 5))
axes = axes.flatten()

for i, (_, row) in enumerate(tomato_classes.iterrows()):
    class_id = row["class_id"]
    idx = label_to_idx.get(class_id)
    if idx is None:
        continue
    img = ds[idx]["image"]
    color = "#2ecc71" if row["is_healthy"] else "#e74c3c"
    axes[i].imshow(img)
    axes[i].set_title(row["disease"], fontsize=8, color=color)
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Томат: все классы PlantVillage", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Размеры изображений

In [ ]:
sample_indices = random.sample(range(N), min(SAMPLE_N, N))
sample_indices.sort()

widths, heights = [], []
for idx in sample_indices:
    img = ds[idx]["image"]
    widths.append(img.width)
    heights.append(img.height)

widths  = np.array(widths)
heights = np.array(heights)

print(f"Сэмпл: {len(widths)} изображений")
print(f"Ширина  — min: {widths.min()}, max: {widths.max()}, mean: {widths.mean():.1f}, unique: {len(np.unique(widths))}")
print(f"Высота  — min: {heights.min()}, max: {heights.max()}, mean: {heights.mean():.1f}, unique: {len(np.unique(heights))}")
print(f"Все одинакового размера: {widths.min() == widths.max() and heights.min() == heights.max()}")

# Распределение соотношений сторон
aspect = widths / heights
print(f"Соотношение сторон — min: {aspect.min():.3f}, max: {aspect.max():.3f}, уникальных: {len(np.unique(aspect))}")

In [ ]:
if len(np.unique(widths)) > 1 or len(np.unique(heights)) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist(widths, bins=30, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Ширина (px)")
    axes[0].set_title("Распределение ширин")

    axes[1].hist(heights, bins=30, color="coral", edgecolor="white")
    axes[1].set_xlabel("Высота (px)")
    axes[1].set_title("Распределение высот")

    axes[2].scatter(widths, heights, alpha=0.3, s=10, color="purple")
    axes[2].set_xlabel("Ширина (px)")
    axes[2].set_ylabel("Высота (px)")
    axes[2].set_title("Ширина vs Высота")

    plt.suptitle("Размеры изображений", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"Все изображения одного размера: {widths[0]} x {heights[0]} px")
    # Показываем пример с реальным масштабом
    img = ds[0]["image"]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(img)
    ax.set_title(f"Пример: {widths[0]}x{heights[0]}px  |  {CLASS_NAMES[ds[0]['label']]}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Цветовой анализ

In [ ]:
# RGB гистограммы по сэмплу
all_pixels_r, all_pixels_g, all_pixels_b = [], [], []

for idx in sample_indices:
    img = ds[idx]["image"].convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0
    all_pixels_r.append(arr[:, :, 0].mean())
    all_pixels_g.append(arr[:, :, 1].mean())
    all_pixels_b.append(arr[:, :, 2].mean())

all_pixels_r = np.array(all_pixels_r)
all_pixels_g = np.array(all_pixels_g)
all_pixels_b = np.array(all_pixels_b)

print("Средние значения каналов по датасету:")
print(f"  R: {all_pixels_r.mean():.3f} ± {all_pixels_r.std():.3f}")
print(f"  G: {all_pixels_g.mean():.3f} ± {all_pixels_g.std():.3f}")
print(f"  B: {all_pixels_b.mean():.3f} ± {all_pixels_b.std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, channel, color, name in zip(
    axes,
    [all_pixels_r, all_pixels_g, all_pixels_b],
    ["red", "green", "blue"],
    ["R", "G", "B"],
):
    ax.hist(channel, bins=40, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(channel.mean(), color="black", linestyle="--", linewidth=1.2,
               label=f"mean={channel.mean():.3f}")
    ax.set_xlabel(f"Среднее значение канала {name}")
    ax.set_title(f"Канал {name}")
    ax.legend()

fig.suptitle(f"Распределение средних значений RGB-каналов (сэмпл {SAMPLE_N})", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Средние каналы: здоровые vs больные
sample_labels = [ds[i]["label"] for i in sample_indices]
sample_is_healthy = np.array(["healthy" in CLASS_NAMES[l].lower() for l in sample_labels])

groups = {
    "Здоровые":  sample_is_healthy,
    "Больные":   ~sample_is_healthy,
}
channels = {"R": all_pixels_r, "G": all_pixels_g, "B": all_pixels_b}

print(f"{'Группа':<15} {'R mean':>8} {'G mean':>8} {'B mean':>8} {'Яркость':>10}")
print("-" * 55)
for group_name, mask in groups.items():
    r = all_pixels_r[mask].mean()
    g = all_pixels_g[mask].mean()
    b = all_pixels_b[mask].mean()
    bright = (r + g + b) / 3
    print(f"{group_name:<15} {r:>8.3f} {g:>8.3f} {b:>8.3f} {bright:>10.3f}")

In [ ]:
# Средний R/G/B по каждому растению
plant_color = {}
for idx, lbl in zip(sample_indices, sample_labels):
    plant, _, _ = parse_class(CLASS_NAMES[lbl])
    if plant not in plant_color:
        plant_color[plant] = {"R": [], "G": [], "B": []}

for i, (idx, lbl) in enumerate(zip(sample_indices, sample_labels)):
    plant, _, _ = parse_class(CLASS_NAMES[lbl])
    plant_color[plant]["R"].append(all_pixels_r[i])
    plant_color[plant]["G"].append(all_pixels_g[i])
    plant_color[plant]["B"].append(all_pixels_b[i])

plant_names = sorted(plant_color.keys())
r_means = [np.mean(plant_color[p]["R"]) for p in plant_names]
g_means = [np.mean(plant_color[p]["G"]) for p in plant_names]
b_means = [np.mean(plant_color[p]["B"]) for p in plant_names]

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(plant_names))
w = 0.28
ax.bar(x - w, r_means, w, label="R", color="red",   alpha=0.75)
ax.bar(x,     g_means, w, label="G", color="green", alpha=0.75)
ax.bar(x + w, b_means, w, label="B", color="blue",  alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(plant_names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Среднее значение канала (0–1)")
ax.set_title("Средние RGB-каналы по культурам")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Ключевые выводы

| Характеристика | Значение |
|---|---|
| Всего образцов | см. вывод ячейки 1 |
| Классов | 38 |
| Культур | 14 |
| Болезней/повреждений | 26 уникальных типов |

### Баланс
- Датасет **умеренно несбалансирован**: Томат представлен несравнимо большим числом классов и изображений (10 классов), тогда как Малина, Черника, Соя представлены лишь одним здоровым классом.
- Соотношение здоровые/больные — примерно **1:3**, то есть больных изображений значительно больше.
- Некоторые растения (Апельсин) присутствуют только в виде больного класса — без здоровых образцов.

### Изображения
- Изображения типично **квадратные** (~256×256 px) в трёх каналах RGB.
- Доминирует **зелёный канал** — логично, снимки листьев на фоне растений.
- Средняя яркость **больных** листьев несколько ниже, чем **здоровых** — пожелтение, некрозы уменьшают отражение в зелёном канале.

### Выводы для FL
- При Dirichlet-партиционировании с малым alpha (< 0.5) клиенты будут видеть сильно разные подмножества болезней — реалистичная Non-IID ситуация.
- Для FL-эксперимента рекомендуется нормализация данных по рассчитанным здесь mean/std каналов.
- Несбалансированность по классам требует взвешенного loss или стратифицированного партиционирования.